# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

### 🏗️ work in progress
Voice Chat for Tech Support:
1. Choose your LLM provider and get streaming responses: OpenAI, Anthropic, Google, Ollama
2. Voice Chat, different voice for different model (with some exception 🥴)
3. Voice on/off option: (Be patient; slow if you don't limit response)
4. Upload an image, audio, or type text
5. Tools includes Describe image uploaded 

Disclosure: I started code with _kajalcodes WEEK2 EXERCISE.ipynb (thanks kajal)

## 1. Imports

In [ ]:
# imports 

import os
import json  # parse tool arguments from the model
from dotenv import load_dotenv
from openai import OpenAI
import anthropic

import gradio as gr  # web UI for the chat app

# Some imports for handling images
import base64
from io import BytesIO
from PIL import Image

from datetime import datetime, timezone
import tempfile  # save TTS audio to a temp file for Gradio

## 2. Load Environment Variables

In [ ]:
# Constants

MODEL_GPT = 'gpt-4.1-mini'
MODEL_ANTHROPIC = 'claude-sonnet-4-5-20250929'
MODEL_GOOGLE = 'gemini-2.5-flash-lite'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

In [ ]:
# Connect to Providers: OpenAI, Anthropic, Google, and Ollama

openai_client = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
anthropic_client = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)

google_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_client = OpenAI(api_key=google_api_key, base_url=google_url)

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
# System Prompt

SYSTEM_PROMPT = """You are a patient technical tutor. Explain clearly and accurately.
Use short sections or bullet points when helpful. If you show code, keep it minimal. 
If the user asks for the current time or a calculation, use the available tools."""

## 3. Image Description Function

In [ ]:
# Tool 1 — real clock time (GPT can call this when user asks "what time is it?")

def get_current_time(timezone_name="UTC"):
    from zoneinfo import ZoneInfo
    try:
        tz = ZoneInfo(timezone_name)
    except Exception:
        tz = timezone.utc  # fall back if timezone name is invalid
    now = datetime.now(tz)
    return now.strftime(f"Current time in {timezone_name}: %Y-%m-%d %H:%M:%S %Z")

# Tool 2 — simple calculator (GPT can call this for math questions)

def calculate(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not all(c in allowed for c in expression):
        return "Invalid expression"
    return str(eval(expression))  # ok for a learning demo; not for production

# Tool 3 — vision description (GPT can call this when the user references an uploaded image)

def describe_image(image_path: str) -> str: 
    import os
    if not image_path or not os.path.exists(image_path):
        return f"No image found at path: {image_path}"
    try:
        with open(image_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")
        response = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": "Describe this image in detail."},
                    {"type": "image_url",
                     "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                ],
            }],
            max_tokens=300,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Could not describe image: {e}"
    

In [ ]:
# Tool schemas 

get_current_time_tool = {
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "Get the current date and time in a timezone.",
        "parameters": {
            "type": "object",
            "properties": {
                "timezone_name": {
                    "type": "string",
                    "description": "IANA timezone, e.g. UTC or Europe/London",
                }
            },
            "required": [],
            "additionalProperties": False,
        },
    },
}

calculate_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Evaluate a math expression, e.g. 17 * 23",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Math expression to evaluate",
                }
            },
            "required": ["expression"],
            "additionalProperties": False,
        },
    },
}

describe_image_tool = {
    "type": "function",
    "function": {
        "name": "describe_image",
        "description": (
            "Describe the contents of an image file in detail. "
            "Use this when the user references, uploads, or asks about an image "
            "and you need to know what it depicts."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "image_path": {
                    "type": "string",
                    "description": "Local filesystem path to the image file to describe.",
                }
            },
            "required": ["image_path"],
        },
    },
}


TOOLS = [get_current_time_tool, calculate_tool, describe_image_tool]

## 4. Tool Calling

In [ ]:
# Run whichever tool GPT asked for, then send results back to the model

def handle_tool_calls(message):
    results = []
    for tc in message.tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments or "{}")
        if name == "get_current_time":
            out = get_current_time(**args)
        elif name == "calculate":
            out = calculate(**args)
        elif name == "describe_image":
            out = "describe_image(**args)"  ### image
        else:
            out = f"Unknown tool: {name}"
        print(f"Tool called: {name}({args})", flush=True)
        results.append({"role": "tool", "content": out, "tool_call_id": tc.id})
    return results

## 5. Audio Transcription Function

In [ ]:
# Use only OpenAI models here ..

# Transcribe (Speech -> Text)
def transcribe_audio(audio_path):
    """Mic recording → text via Whisper."""
    if not audio_path:
        return ""
    with open(audio_path, "rb") as f:
        transcript = openai_client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
        )
    print(f"Transcribed: {transcript.text}", flush=True)
    return transcript.text

# Talker (Text -> Speech) (Optional)
def text_to_speech(model_choice, text):
    """Answer text → spoken audio bytes."""

    # assign voice to model (currently hard-coded)
    if model_choice == "Llama 3.2":
        model_voice = "alloy" 
    elif model_choice == "claude-sonnet-4-5-20250929":
        model_voice = "echo"
    elif model_choice == "gemini-2.5-flash-lite":
        model_voice = "fable"
    else:
        model_voice = "onyx"

    if not text:
        return None
    response = openai_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice=model_voice,  # Try voices: alloy, echo, fable, onyx, nova, shimmer
        input=text[:4096],  # shorten long answers for TTS
    )
    return response.content


def save_audio_bytes(audio_bytes):
    """Gradio needs a file path to play audio."""
    f = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    f.write(audio_bytes)
    f.close()
    return f.name

## 6. LLM Provider Functions

In [ ]:
# Currently hard-coding models

def chat_stream(message, history, model_choice):
    # Route to the right client based on dropdown
    if model_choice == "Llama 3.2":
        client = ollama_client
        model = MODEL_LLAMA
    elif model_choice == "claude-sonnet-4-5-20250929":
        client = anthropic_client
        model = MODEL_ANTHROPIC
    elif model_choice == "gemini-2.5-flash-lite":
        client = google_client
        model = MODEL_GOOGLE
    else:
        client = openai_client
        model = MODEL_GPT

    print(f"Calling model: {model} (selected: {model_choice})", flush=True)

    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + history
        + [{"role": "user", "content": message}]
    )

    # Claude, Google, Ollama path — stream tokens one by one (no tools)
    if model_choice in ("claude-sonnet-4-5-20250929", "gemini-2.5-flash-lite", "Llama 3.2"):
        stream = client.chat.completions.create(
            model=model, messages=messages, stream=True
        )
        response_text = ""
        for chunk in stream:
            response_text += chunk.choices[0].delta.content or ""
            yield response_text  # Gradio updates the chat as text grows
        return


    # GPT path — may call tools first, then return final answer
    response = client.chat.completions.create(
        model=model, messages=messages, tools=TOOLS
    )
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        messages.append(msg)
        messages.extend(handle_tool_calls(msg))
        response = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS
        )

    final = response.choices[0].message.content or ""
    yield final

## 7.  Gradio Returns

In [ ]:
# Collects whatever the user gave (text, speech, image, sends it to the LLM, then returns both text reply and optionally, a spoken version)

def respond(message, history, model_choice, image_file, audio_file, voice_reply):
    history = history or []

    # If user recorded audio, transcribe audio to text
    if audio_file:
        message = transcribe_audio(audio_file) or message
    if not message or not message.strip():
        return "", history, None

    # if user uploaded an image,
    if image_file is not None:
        description = describe_image(image_file)
        message = message + f"\n\n[The user uploaded an image on {image_file}:\n {description}]"

    history = history + [{"role": "user", "content": message.strip()}]
    full = ""

    # Stream responses from provider
    for chunk in chat_stream(message.strip(), history[:-1], model_choice):
        full = chunk
        yield "", history + [{"role": "assistant", "content": full}], None

    # Optionally speak the final answer aloud
    audio_path = None
    if voice_reply and full:
        audio_bytes = text_to_speech(model_choice, full)
        if audio_bytes:
            audio_path = save_audio_bytes(audio_bytes)
    yield "", history + [{"role": "assistant", "content": full}], audio_path

## 8. Create Gradio Interface

In [ ]:
# Create Gradio User Interface

with gr.Blocks(title="Voice Chatbot") as ui:
    gr.Markdown("# Technical Support")
    gr.Markdown("Upload an image, audio, or type text. Choose your LLM provider and get streaming responses.")

    with gr.Row():
        model_dropdown = gr.Dropdown(
            ["gpt-4.1-mini", "claude-sonnet-4-5-20250929", "gemini-2.5-flash-lite", "Llama 3.2"],
            value="gpt-4.1-mini",
            label="Model",
        )

    with gr.Row():
        chatbot = gr.Chatbot(height=350, type="messages", label="Conversation")
        image_input = gr.Image(height=350, type="filepath", label="Upload an image")

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Type here ...     or start by clicking on example below.",
            scale=4,
            label="Type your question",
        )
        send = gr.Button("Send", variant="primary", scale=1)
        clear = gr.Button("Clear", scale=1)

    with gr.Row():
        audio_input = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="Or speak your question",
          )
        with gr.Row():
            # Voice reply — option to turn off voice
            voice_reply = gr.Checkbox(value=True, label="Voice on/off")

    with gr.Row():
        # Spoken response — autoplay makes it speak automatically
        audio_output = gr.Audio(label="Spoken Response", autoplay=True)

    # One-click test questions (includes Week 1 example)
    gr.Examples(
        examples=[
            ["What time is it in UTC?"],
            ["Please describe the image I uploaded"],
            ["Explain the Transformer architecture to an aspiring AI engineer"],
            [
                'Please explain what this code does and why:\n'
                'yield from {book.get("author") for book in books if book.get("author")}'
            ]

        ],
        inputs=msg,
    )

    inputs = [msg, chatbot, model_dropdown, image_input, audio_input, voice_reply]
    outputs = [msg, chatbot, audio_output]

    msg.submit(respond, inputs=inputs, outputs=outputs)
    send.click(respond, inputs=inputs, outputs=outputs)
    clear.click(lambda: ([], None, None), outputs=[chatbot, audio_output, image_input])

## 9. Launch

In [ ]:
ui.launch(inbrowser=True)